# LLaMAC PPG Coherent Split R2 Model Search

이 노트북은 직전 R2 > 0.3 실험의 `window_random` split을 더 정합적인 split으로 바꿔 모델을 다시 찾습니다.

정합성 기준:
- group key = `SubjectID + Trial`
- 같은 trial에서 나온 rolling window는 train / validation / test 중 하나에만 들어갑니다.
- validation set으로 모델을 고르고, test set은 최종 선택 후 한 번만 평가합니다.
- 기본 평가는 trial-level입니다. 각 trial의 window 예측을 평균한 뒤 하나의 trial label과 비교합니다.

결론적으로 이 split에서는 R2가 0.3을 넘지 않습니다. 가장 높은 모델과 점수를 그대로 남겨 다음 연구 판단에 쓰도록 합니다.

In [ ]:
import json
import pickle
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
try:
    import lightgbm as lgb
except Exception:
    lgb = None
try:
    import xgboost as xgb
except Exception:
    xgb = None

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = Path.cwd().resolve()

RANDOM_STATE = 42
TARGETS = ["Valence", "Arousal", "Dominance", "Liking", "Emotion_Intensity"]
META = {"SubjectID", "Trial", "WindowStartSec"}
FEATURE_PATH = PROJECT_ROOT / "artifacts/results/ppg_window_r2/features_p40_w30_s5.parquet"
SUMMARY_PATH = PROJECT_ROOT / "artifacts/results/ppg_window_r2/coherent_trial_aggregate_search.json"
MODEL_PATH = PROJECT_ROOT / "artifacts/models/ppg_window_extra_trees_coherent_trial_group.pkl"
SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

if not FEATURE_PATH.exists():
    raise FileNotFoundError(
        f"Missing feature cache: {FEATURE_PATH}. Run notebooks/LLaMAC_PPG_R2_Regression_Model.ipynb first."
    )
print(FEATURE_PATH)

## 1. Load PPG window features and make coherent splits

In [ ]:
data = pd.read_parquet(FEATURE_PATH).replace([np.inf, -np.inf], np.nan)
feature_cols = [
    col for col in data.columns
    if col not in META and col not in TARGETS and data[col].notna().any()
]
X = data[feature_cols]
y = data[TARGETS]
groups = (data["SubjectID"].astype(str) + "_" + data["Trial"].astype(str)).to_numpy()

outer = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
trainval_idx, test_idx = next(outer.split(X, y, groups))
inner = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_rel, val_rel = next(inner.split(X.iloc[trainval_idx], y.iloc[trainval_idx], groups[trainval_idx]))
train_idx = trainval_idx[train_rel]
val_idx = trainval_idx[val_rel]

split_report = {
    "rows": {"train": len(train_idx), "validation": len(val_idx), "test": len(test_idx)},
    "groups": {
        "train": int(len(np.unique(groups[train_idx]))),
        "validation": int(len(np.unique(groups[val_idx]))),
        "test": int(len(np.unique(groups[test_idx]))),
    },
    "overlap_checks": {
        "train_validation": len(set(groups[train_idx]) & set(groups[val_idx])),
        "train_test": len(set(groups[train_idx]) & set(groups[test_idx])),
        "validation_test": len(set(groups[val_idx]) & set(groups[test_idx])),
    },
}
print(json.dumps(split_report, indent=2, ensure_ascii=False))
assert split_report["overlap_checks"] == {"train_validation": 0, "train_test": 0, "validation_test": 0}

## 2. Candidate models

Validation selection metric is mean trial-level R2 across all five targets. Trial-level prediction means averaging all window predictions inside a held-out trial before computing R2.

In [ ]:
def make_pipeline(model, scale=False):
    steps = [("imputer", SimpleImputer(strategy="median")), ("variance", VarianceThreshold())]
    if scale:
        steps.append(("scale", StandardScaler()))
    steps.append(("model", model))
    return Pipeline(steps)

candidates = []
for n_estimators in [300, 600, 1000]:
    for max_features in [0.7, 0.9, 1.0]:
        candidates.append((
            f"ExtraTrees_n{n_estimators}_mf{max_features}_leaf1",
            make_pipeline(ExtraTreesRegressor(
                n_estimators=n_estimators,
                max_features=max_features,
                min_samples_leaf=1,
                bootstrap=False,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ))
for n_estimators in [500, 800]:
    candidates.append((
        f"ExtraTrees_n{n_estimators}_sqrt_leaf1",
        make_pipeline(ExtraTreesRegressor(
            n_estimators=n_estimators,
            max_features="sqrt",
            min_samples_leaf=1,
            bootstrap=False,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ))
candidates.extend([
    ("RandomForest_n500_mf0.7", make_pipeline(RandomForestRegressor(n_estimators=500, max_features=0.7, min_samples_leaf=1, random_state=RANDOM_STATE, n_jobs=-1))),
    ("RandomForest_n500_mf1.0", make_pipeline(RandomForestRegressor(n_estimators=500, max_features=1.0, min_samples_leaf=1, random_state=RANDOM_STATE, n_jobs=-1))),
    ("HistGB_700_lr02_leaf31", make_pipeline(MultiOutputRegressor(HistGradientBoostingRegressor(max_iter=700, learning_rate=0.02, max_leaf_nodes=31, l2_regularization=0.01, random_state=RANDOM_STATE), n_jobs=-1))),
    ("RidgeCV", make_pipeline(RidgeCV(alphas=np.logspace(-3, 3, 13)), scale=True)),
])
if lgb is not None:
    for leaves in [31, 63, 127]:
        candidates.append((f"LightGBM_1000_lr02_l{leaves}", make_pipeline(MultiOutputRegressor(lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.02, num_leaves=leaves, subsample=0.9, colsample_bytree=0.9, random_state=RANDOM_STATE, n_jobs=1, verbose=-1), n_jobs=-1))))
if xgb is not None:
    candidates.append(("XGBoost_800_d6", make_pipeline(MultiOutputRegressor(xgb.XGBRegressor(n_estimators=800, learning_rate=0.02, max_depth=6, subsample=0.9, colsample_bytree=0.9, tree_method="hist", random_state=RANDOM_STATE, n_jobs=1, verbosity=0), n_jobs=-1))))
print([name for name, _ in candidates])

In [ ]:
def window_metrics(idx, pred):
    rows = []
    for i, target in enumerate(TARGETS):
        truth = y.iloc[idx][target]
        estimate = pred[:, i]
        rows.append({
            "target": target,
            "r2": float(r2_score(truth, estimate)),
            "rmse": float(mean_squared_error(truth, estimate) ** 0.5),
            "mae": float(mean_absolute_error(truth, estimate)),
        })
    return rows


def trial_metrics(idx, pred):
    pred_frame = pd.DataFrame(pred, columns=TARGETS)
    pred_frame["group"] = groups[idx]
    truth_frame = y.iloc[idx].copy().reset_index(drop=True)
    truth_frame["group"] = groups[idx]
    pred_by_trial = pred_frame.groupby("group")[TARGETS].mean().sort_index()
    truth_by_trial = truth_frame.groupby("group")[TARGETS].first().loc[pred_by_trial.index]
    rows = []
    for target in TARGETS:
        rows.append({
            "target": target,
            "r2": float(r2_score(truth_by_trial[target], pred_by_trial[target])),
            "rmse": float(mean_squared_error(truth_by_trial[target], pred_by_trial[target]) ** 0.5),
            "mae": float(mean_absolute_error(truth_by_trial[target], pred_by_trial[target])),
        })
    return rows

search_results = []
best = None
for name, candidate in candidates:
    start = time.time()
    candidate.fit(X.iloc[train_idx], y.iloc[train_idx])
    val_pred = candidate.predict(X.iloc[val_idx])
    val_trial = trial_metrics(val_idx, val_pred)
    val_window = window_metrics(val_idx, val_pred)
    row = {
        "name": name,
        "validation_trial_mean_r2": float(np.mean([m["r2"] for m in val_trial])),
        "validation_trial_min_r2": float(np.min([m["r2"] for m in val_trial])),
        "validation_trial_metrics": val_trial,
        "validation_window_metrics": val_window,
        "fit_seconds": float(time.time() - start),
    }
    search_results.append(row)
    print(f"{name:30s} trial_mean={row['validation_trial_mean_r2']:.4f} trial_min={row['validation_trial_min_r2']:.4f} sec={row['fit_seconds']:.1f}")
    if best is None or row["validation_trial_mean_r2"] > best["validation_trial_mean_r2"]:
        best = row

leaderboard = pd.DataFrame([
    {"model": r["name"], "trial_mean_r2": r["validation_trial_mean_r2"], "trial_min_r2": r["validation_trial_min_r2"], "fit_seconds": r["fit_seconds"]}
    for r in search_results
]).sort_values("trial_mean_r2", ascending=False)
display(leaderboard)
print("BEST", best["name"])


## 3. Refit best model and evaluate test once

In [ ]:
candidate_map = dict(candidates)
selected_model = candidate_map[best["name"]]
selected_model.fit(X.iloc[trainval_idx], y.iloc[trainval_idx])
test_pred = selected_model.predict(X.iloc[test_idx])
test_trial = trial_metrics(test_idx, test_pred)
test_window = window_metrics(test_idx, test_pred)

test_trial_df = pd.DataFrame(test_trial)
test_window_df = pd.DataFrame(test_window)
print("Trial-level test metrics")
display(test_trial_df)
print("Window-level secondary test metrics")
display(test_window_df)

summary = {
    "split": "trial_grouped_train_validation_test",
    "evaluation_primary": "trial_mean_predictions",
    "random_state": RANDOM_STATE,
    "feature_path": str(FEATURE_PATH.relative_to(PROJECT_ROOT)),
    "rows": int(len(data)),
    "features": int(len(feature_cols)),
    "total_groups": int(len(np.unique(groups))),
    "split_report": split_report,
    "selection_metric": "validation trial-level mean R2 across five targets",
    "best_model": best["name"],
    "validation": best,
    "test_trial_metrics": test_trial,
    "test_window_metrics": test_window,
    "all_results": search_results,
    "interpretation": "This coherent trial-group split removes same-trial window leakage. It lowers R2 versus window_random; the selected model is the best validation performer found here, not an R2 > 0.3 result.",
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
with MODEL_PATH.open("wb") as f:
    pickle.dump({
        "model": selected_model,
        "feature_cols": feature_cols,
        "target_cols": TARGETS,
        "summary": summary,
    }, f)
print(SUMMARY_PATH)
print(MODEL_PATH)
print("test_trial_mean_r2", float(test_trial_df["r2"].mean()))
print("test_trial_min_r2", float(test_trial_df["r2"].min()))